# Exercise 8: a probabilistic positional motif and a 1D CNN

Generate 1,000-nt sequences with one planted instance of **motif A**, an ATG-containing motif, and one of **motif B**, a TATA-containing motif. Sample each instance from a position-specific probability matrix; strings vary between sequences. The ATG core is 99.9% conserved at each base, and the TATAAA core is 97% conserved at each base. Flanks are less conserved.

- **Class 0:** place B uniformly among all full-length positions that do not overlap A. It may fall upstream or downstream, including in the class-1 region by chance.
- **Class 1:** concentrate B upstream of A, with a curve peaked at −50 nt and supported on −100…−30 nt.

Define offset as **the nominal TATA core start minus the nominal A of ATG**, using zero-based coordinates. It is not an inter-motif gap. Both classes use the same motif probabilities, background and ATG position distribution. ATG positions leave room for the full positive offset range.

Labels record the generating class, not a hard spacing threshold. Random class-0 placements can resemble class 1, motif instances vary, and incidental background matches are retained. These introduce ambiguity without arbitrary label flips.

This is an artificial pattern-recognition task. TATA boxes concern transcription initiation; ATG is a translation start codon, not a transcription start site. These labels are not a biological rule for identifying functional promoters or translated genes.

The notebook generates and saves its own CSV, introduces one-hot encoding, and trains a small convolutional neural network (CNN). No earlier notebook or upload is needed. **Run all includes the final test evaluation.**


## 1. Imports and settings

PyTorch is normally installed in Colab. For local use, install the updated requirements as described in the README. We use a GPU when available; this small network also runs on a CPU.

Make 10,000 sequences per generating class. Change `ofile` to choose the CSV path.


In [ ]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, log_loss,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)

seed = 42
n_positive = 10_000
n_negative = 10_000
sequence_length = 1_000
max_epochs = 25
batch_size = 128
learning_rate = 0.003
ofile = "exercise_8_output/dataset.csv"

bases = np.array(list("ACGT"))
torch.manual_seed(seed)
torch.set_num_threads(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 2. Define motif probability matrices

As in Exercise 1, rows are A, C, G, T and columns are positions. Normalize each column of these illustrative count matrices to obtain sampling probabilities. These are not experimentally estimated motifs.

A has consensus **GCCACCATGG**, with ATG at columns 6–8 (zero-based). B has consensus **CTATAAAGGC**, with TATAAA at columns 1–6. Both motifs are 10 nt long. Nominal core anchors stay at these positions even when a sampled base differs.

A probability matrix generates instances. A corresponding PWM contains log-odds scores, log2(probability/0.25), for scoring against uniform background. We display probabilities and calculate the PWMs; generation samples from probabilities.


In [ ]:
motif_a_counts = np.array([
    [ 10,  10,  10, 270,  10,  10, 2997,    1,    1,  10],
    [ 10, 270, 270,  10, 270, 270,    1,    1,    1,  10],
    [270,  10,  10,  10,  10,  10,    1,    1, 2997, 270],
    [ 10,  10,  10,  10,  10,  10,    1, 2997,    1,  10],
])
motif_b_counts = np.array([
    [ 10,   3, 291,   3, 291, 291, 291,  10,  10,  10],
    [270,   3,   3,   3,   3,   3,   3,  10,  10, 270],
    [ 10,   3,   3,   3,   3,   3,   3, 270, 270,  10],
    [ 10, 291,   3, 291,   3,   3,   3,  10,  10,  10],
])
motif_a_probabilities = motif_a_counts / motif_a_counts.sum(axis=0)
motif_b_probabilities = motif_b_counts / motif_b_counts.sum(axis=0)
motif_a_pwm = np.log2(motif_a_probabilities / 0.25)
motif_b_pwm = np.log2(motif_b_probabilities / 0.25)
motif_a_length = motif_a_probabilities.shape[1]
motif_b_length = motif_b_probabilities.shape[1]
atg_core_index = 6
tata_core_index = 1
print("Motif A probabilities:")
print(pd.DataFrame(motif_a_probabilities, index=bases).round(4))
print("\nMotif B probabilities:")
print(pd.DataFrame(motif_b_probabilities, index=bases).round(4))


## 3. Define the class-1 offset curve

The requested support is asymmetric around −50. For integer offsets d from −100 through −30, set p(d) proportional to exp[−0.5 × ((d+50)/sigma(d))²], with sigma(d)=20 for d<−50 and sigma(d)=8 otherwise. Normalize the weights to sum to one.

This discrete, truncated distribution has its mode exactly at −50 and a longer upstream tail. Its mean is somewhat upstream of −50 because the curve is asymmetric. We sample directly from this distribution, without clipping normal draws at the boundaries.

Both classes use ATG anchors from 101–996 inclusive, leaving room for all positive offsets and the full A motif.


In [ ]:
offset_values = np.arange(-100, -29)
offset_widths = np.where(offset_values < -50, 20.0, 8.0)
offset_probabilities = np.exp(-0.5 * ((offset_values + 50) / offset_widths) ** 2)
offset_probabilities /= offset_probabilities.sum()
min_atg_position = int(tata_core_index - offset_values.min())
max_atg_position = sequence_length - (motif_a_length - atg_core_index)
print("Class-1 offset mode:", offset_values[np.argmax(offset_probabilities)])
print("Class-1 offset mean:", round(np.sum(offset_values * offset_probabilities), 2))
print("Shared ATG anchor range:", min_atg_position, "to", max_atg_position)


## 4. Generate backgrounds, sample motifs and plant them

Sample independent background bases with A=C=G=T=25%, and sample motif columns independently from the probability matrices. Plant one instance of each motif by replacing bases on the supplied strand, keeping sequence length unchanged.

For class 0, sample B uniformly among positions that fit and do not overlap planted A. For class 1, sample its core offset from the curve. Retain chance background matches; only the two planted instances have known positions. A sampled motif need not equal its consensus string.

Each sequence has its own independently sampled background. Both motif distributions are identical across classes.


In [ ]:
rng = np.random.default_rng(seed)
n_sequences = n_negative + n_positive
labels = np.repeat([0, 1], [n_negative, n_positive])
backgrounds = rng.choice(bases, size=(n_sequences, sequence_length))
motif_a_instances = np.empty((n_sequences, motif_a_length), dtype="U1")
motif_b_instances = np.empty((n_sequences, motif_b_length), dtype="U1")
for position in range(motif_a_length):
    motif_a_instances[:, position] = rng.choice(
        bases, size=n_sequences, p=motif_a_probabilities[:, position],
    )
for position in range(motif_b_length):
    motif_b_instances[:, position] = rng.choice(
        bases, size=n_sequences, p=motif_b_probabilities[:, position],
    )

atg_positions = rng.integers(min_atg_position, max_atg_position + 1, size=n_sequences)
possible_b_starts = np.arange(sequence_length - motif_b_length + 1)
rows = []
for i, label in enumerate(labels):
    atg_position = int(atg_positions[i])
    a_start = atg_position - atg_core_index
    if label == 0:
        nonoverlapping = ((possible_b_starts + motif_b_length <= a_start)
                          | (possible_b_starts >= a_start + motif_a_length))
        b_start = int(rng.choice(possible_b_starts[nonoverlapping]))
    else:
        offset = int(rng.choice(offset_values, p=offset_probabilities))
        b_start = atg_position + offset - tata_core_index
    tata_position = b_start + tata_core_index

    backgrounds[i, a_start:a_start + motif_a_length] = motif_a_instances[i]
    backgrounds[i, b_start:b_start + motif_b_length] = motif_b_instances[i]
    rows.append({
        "sequence": "".join(backgrounds[i]), "label": int(label),
        "atg_start": atg_position, "tata_start": tata_position,
        "motif_a_start": a_start, "motif_b_start": b_start,
        "offset": tata_position - atg_position,
        "motif_a_instance": "".join(motif_a_instances[i]),
        "motif_b_instance": "".join(motif_b_instances[i]),
    })

dataset = pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)
print("Sequences:", len(dataset))
print(dataset["label"].value_counts())
dataset.head()


## 5. Check, save and inspect the distributions

The CSV stores the sequence, generating label, nominal core positions, full motif starts, offset and sampled motif strings. These columns let us check the data, but **the CNN receives only the sequence**.

The first histogram shows both classes across the full offset range. The second compares sampled class-1 offsets with the specified curve. The class-0 overlap fraction shows why perfect classification is not the goal.


In [ ]:
assert len(dataset) == n_sequences
assert dataset["sequence"].str.fullmatch(f"[ACGT]{{{sequence_length}}}").all()
assert dataset["sequence"].is_unique
assert dataset["offset"].eq(dataset["tata_start"] - dataset["atg_start"]).all()
positive_offsets = dataset.loc[dataset["label"] == 1, "offset"]
negative_offsets = dataset.loc[dataset["label"] == 0, "offset"]
assert positive_offsets.between(-100, -30).all()
print("Class-0 fraction in the class-1 offset range:",
      round(negative_offsets.between(-100, -30).mean(), 4))

output_path = Path(ofile)
output_path.parent.mkdir(parents=True, exist_ok=True)
dataset.to_csv(output_path, index=False)
print("Saved:", output_path)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for label in (0, 1):
    offsets = dataset.loc[dataset["label"] == label, "offset"]
    axes[0].hist(offsets, bins=np.arange(-1000, 1001, 20), density=True,
                 histtype="step", linewidth=2, label=f"Class {label}")
axes[0].set(xlabel="TATA core start − ATG start (nt)", ylabel="Density",
            title="Planted relative positions")
axes[0].legend()
axes[1].hist(positive_offsets, bins=np.arange(-100.5, -28.5, 1), density=True,
             alpha=0.5, label="Sampled class 1")
axes[1].plot(offset_values, offset_probabilities, color="black", label="Specified curve")
axes[1].set(xlabel="TATA core start − ATG start (nt)", ylabel="Probability per nt",
            title="Class 1: peak at −50 nt")
axes[1].legend()
plt.tight_layout()
plt.show()


## 6. Reserve training, validation and test sequences

Within each label, reserve 10% for test and 20% for validation, using the original class size for both counts. The remaining 70% is training data.

Choose the CNN epoch using validation loss. Leave the test sequences unused until the final evaluation cell.


In [ ]:
split_rng = np.random.default_rng(seed)
test_indices = []
validation_indices = []
for label, group in dataset.groupby("label"):
    indices = split_rng.permutation(group.index)
    n_test = int(0.10 * len(group))
    n_validation = int(0.20 * len(group))
    test_indices.extend(indices[:n_test])
    validation_indices.extend(indices[n_test:n_test + n_validation])

test_data = dataset.loc[test_indices].copy()
validation_data = dataset.loc[validation_indices].copy()
train_data = dataset.drop(test_indices + validation_indices).copy()
print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Test (reserved):", len(test_data))


## 7. Represent DNA with one-hot encoding

Use A=[1,0,0,0], C=[0,1,0,0], G=[0,0,1,0], and T=[0,0,0,1]. Each sequence becomes a 1000-by-4 array. This preserves order without treating base IDs as meaningful numeric values.

PyTorch's `Conv1d` expects **batch, channels, length**, so we rearrange the axes to N-by-4-by-1000. The encoding is fixed: there is no transformation to fit.


In [ ]:
def encode_sequences(sequences):
    base_ids = {"A": 0, "C": 1, "G": 2, "T": 3}
    ids = np.array([[base_ids[base] for base in sequence] for sequence in sequences])
    encoded = np.eye(4, dtype=np.float32)[ids]
    return torch.from_numpy(encoded).permute(0, 2, 1).contiguous()

print("ACGT one-hot rows:")
print(encode_sequences(["ACGT"])[0].T)
x_train = encode_sequences(train_data["sequence"])
y_train = torch.tensor(train_data["label"].to_numpy(), dtype=torch.float32)
x_validation = encode_sequences(validation_data["sequence"]).to(device)
y_validation = validation_data["label"].to_numpy()
validation_targets = torch.tensor(y_validation, dtype=torch.float32, device=device)
train_loader = DataLoader(
    TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True,
    generator=torch.Generator().manual_seed(seed),
)
print("Training tensor:", tuple(x_train.shape))
print("Validation tensor:", tuple(x_validation.shape))


## 8. Build a CNN that combines local motif responses

Use the two supplied log-odds PWMs as fixed 10-base convolution filters. On one-hot DNA, each filter's dot product is exactly its PWM score for that window. These are known motif scanners; the model learns the positional combination rather than discovering both motifs from scratch.

Scale the scores by 10 and subtract 0.6, so ReLU retains positive responses for scores above 6 bits (a likelihood ratio over 64 relative to background for that window). This response is not a probability. The first convolution has 991 valid windows. Local max pooling groups four positions, preserving 248 ordered bins; `ceil_mode=True` keeps the final partial bin.

A trainable convolution combines 31 adjacent bins into 16 feature maps. Its **133-base receptive field** is 10+3+30×4. The full motif pair spans at most 105 bases over the positive offset range, so both motifs fit inside this field. Different weights at different offsets allow order and spacing to matter.

Only after combining local responses do we take the maximum across the whole sequence. Unlike local pooling, immediately pooling across the entire sequence would discard the relationship between separate motif responses.

The final linear layer returns a logit. `BCEWithLogitsLoss` combines a sigmoid and binary cross-entropy stably; we apply `sigmoid` ourselves when reporting probabilities.


In [ ]:
motif_scanner = nn.Conv1d(4, 2, kernel_size=10)
with torch.no_grad():
    pwm_weights = torch.tensor(np.stack([motif_a_pwm, motif_b_pwm]) / 10, dtype=torch.float32)
    motif_scanner.weight.copy_(pwm_weights)
    motif_scanner.bias.fill_(-0.6)
motif_scanner.requires_grad_(False)

cnn = nn.Sequential(
    motif_scanner,
    nn.ReLU(),
    nn.MaxPool1d(4, ceil_mode=True),
    nn.Conv1d(2, 16, kernel_size=31, padding=15),
    nn.ReLU(),
    nn.AdaptiveMaxPool1d(1),
    nn.Flatten(),
    nn.Linear(16, 1),
).to(device)
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=learning_rate)
print(cnn)
print("Trainable parameters:", sum(p.numel() for p in cnn.parameters() if p.requires_grad))


## 9. Train: forward pass, backpropagation, optimizer step

For each mini-batch: clear old gradients, calculate logits and loss, call `loss.backward()` to compute gradients, then call `optimizer.step()` to update trainable weights. The supplied PWM scanner stays fixed. Backpropagation computes gradients; Adam decides how to use them.

Training loss is the mean mini-batch loss encountered during the epoch, while weights are changing. Validation loss is calculated after the epoch, with fixed weights and gradient tracking disabled. Save the weights whenever validation loss improves; ties keep the earlier epoch. Never update weights with validation examples.


In [ ]:
training_losses = []
validation_losses = []
best_validation_loss = np.inf

for epoch in range(1, max_epochs + 1):
    cnn.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = cnn(batch_x).squeeze(1)
        loss = loss_function(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_y)

    training_loss = total_loss / len(train_data)
    cnn.eval()
    with torch.no_grad():
        validation_logits = cnn(x_validation).squeeze(1)
        validation_loss = loss_function(validation_logits, validation_targets).item()
    training_losses.append(training_loss)
    validation_losses.append(validation_loss)
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_epoch = epoch
        best_state = deepcopy(cnn.state_dict())
    print(f"Epoch {epoch}: training={training_loss:.4f}, validation={validation_loss:.4f}")

cnn.load_state_dict(best_state)
cnn.eval()
print("Selected epoch:", best_epoch)
epochs = range(1, max_epochs + 1)
plt.figure(figsize=(9, 5))
plt.plot(epochs, training_losses, label="Training (during epoch)")
plt.plot(epochs, validation_losses, label="Validation (after epoch)")
plt.axvline(best_epoch, color="black", linestyle="--", label="Selected epoch")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.legend()
plt.tight_layout()
plt.show()


## 10. Evaluate validation performance

Report the familiar eight metrics and use a probability threshold of 0.5. Inspect accuracy and mean predicted probability by generating class and offset region. A class-0 sequence in the concentrated region can legitimately be hard to distinguish from class 1. Position metadata is used only for interpretation.


In [ ]:
with torch.no_grad():
    validation_probabilities = torch.sigmoid(cnn(x_validation).squeeze(1)).cpu().numpy()
validation_predictions = (validation_probabilities >= 0.5).astype(int)
validation_results = pd.Series({
    "accuracy": accuracy_score(y_validation, validation_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_validation, validation_predictions),
    "precision": precision_score(y_validation, validation_predictions, zero_division=0),
    "recall": recall_score(y_validation, validation_predictions, zero_division=0),
    "f1": f1_score(y_validation, validation_predictions, zero_division=0),
    "roc_auc": roc_auc_score(y_validation, validation_probabilities),
    "average_precision": average_precision_score(y_validation, validation_probabilities),
    "log_loss": log_loss(y_validation, validation_probabilities),
}, name="CNN validation")
print(validation_results.round(4).to_string())
validation_groups = validation_data[["label", "offset"]].copy()
validation_groups["in_peak_region"] = validation_groups["offset"].between(-100, -30)
validation_groups["correct"] = (validation_predictions == y_validation).astype(int)
validation_groups["probability"] = validation_probabilities
print("\nValidation by generating class and offset region:")
print(validation_groups.groupby(["label", "in_peak_region"])[["correct", "probability"]]
      .agg(["mean", "count"]).round(4))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
RocCurveDisplay.from_predictions(y_validation, validation_probabilities, ax=axes[0])
axes[0].plot([0, 1], [0, 1], "k--")
PrecisionRecallDisplay.from_predictions(y_validation, validation_probabilities, ax=axes[1])
axes[1].axhline(y_validation.mean(), color="black", linestyle="--")
ConfusionMatrixDisplay.from_predictions(y_validation, validation_predictions, ax=axes[2],
                                       labels=[0, 1], cmap="Blues", colorbar=False)
axes[0].set_title("Validation ROC")
axes[1].set_title("Validation precision-recall")
axes[2].set_title("Validation confusion matrix")
plt.tight_layout()
plt.show()


## 11. STOP before the final test evaluation

Finish selecting the epoch and discussing validation results first. Then evaluate the selected weights once on the reserved test sequences. Do not change the architecture, threshold, or epoch using test results. **Run all includes this cell.**


In [ ]:
x_test = encode_sequences(test_data["sequence"]).to(device)
y_test = test_data["label"].to_numpy()
with torch.no_grad():
    test_probabilities = torch.sigmoid(cnn(x_test).squeeze(1)).cpu().numpy()
test_predictions = (test_probabilities >= 0.5).astype(int)
test_results = pd.Series({
    "accuracy": accuracy_score(y_test, test_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_predictions),
    "precision": precision_score(y_test, test_predictions, zero_division=0),
    "recall": recall_score(y_test, test_predictions, zero_division=0),
    "f1": f1_score(y_test, test_predictions, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_probabilities),
    "average_precision": average_precision_score(y_test, test_probabilities),
    "log_loss": log_loss(y_test, test_probabilities),
}, name="CNN test")
print(test_results.round(4).to_string())
test_groups = test_data[["label", "offset"]].copy()
test_groups["in_peak_region"] = test_groups["offset"].between(-100, -30)
test_groups["correct"] = (test_predictions == y_test).astype(int)
test_groups["probability"] = test_probabilities
print("\nTest by generating class and offset region:")
print(test_groups.groupby(["label", "in_peak_region"])[["correct", "probability"]]
      .agg(["mean", "count"]).round(4))
ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, labels=[0, 1],
                                       cmap="Blues", colorbar=False)
plt.title("Test confusion matrix")
plt.tight_layout()
plt.show()
